In [1]:
# multiturn_stats_report.py
import pandas as pd
import tiktoken
from collections import Counter
import json


# 加载多轮数据
df = pd.read_json("train_final.jsonl", lines=True)
print(f"多轮数据集加载完成：{len(df):,} 条样本\n")
print("="*80)
print("                    MULTI-TURN RAL2M TRAINING SET 完整统计报告")
print("="*80)

# 初始化 tokenizer（GPT系列通用）
enc = tiktoken.get_encoding("cl100k_base")

def count_tokens(text):
    return len(enc.encode(str(text))) if pd.notna(text) else 0

# 1. 基本结构统计
n_samples = len(df)
n_datasets = df['dataset'].nunique()
n_unique_qids = df['Question_ID'].nunique()
n_history_lengths = df['turn_number'].value_counts().sort_index()

print(f"总样本数                  : {n_samples:,}")
print(f"来源数据集数量            : {n_datasets:,}")
print()

print("对话轮数分布 (turn_number):")
for turn, cnt in n_history_lengths.items():
    print(f"   {turn:>2} 轮对话（含当前轮） → {cnt:>6,} 条  ({cnt/n_samples:.1%})")
print()

# 2. Token 长度统计
df['ctx_tokens'] = df['dialogue_context'].apply(count_tokens)
df['query_tokens'] = df['user_query'].apply(count_tokens)
df['answer_tokens'] = df['true_a_i'].apply(count_tokens)

# retrieved_doc_ids → 实际文档文本（从 D2 加载）
print("加载文档库 D2 用于计算检索文档长度...", end=" ")
d2_docs = {}
with open("D2.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)
        d2_docs[obj["doc_id"]] = obj["text"]
print(f"成功加载 {len(d2_docs):,} 条文档")

def get_retrieved_text(doc_ids):
    if not isinstance(doc_ids, list):
        doc_ids = eval(doc_ids) if isinstance(doc_ids, str) else []
    texts = [d2_docs.get(did, "") for did in doc_ids]
    return " ".join(texts)

df['retrieved_text'] = df['retrieved_doc_ids'].apply(get_retrieved_text)
df['retrieved_tokens'] = df['retrieved_text'].apply(count_tokens)

# 3. 汇总 Token 统计
stats = {
    "历史上下文 (dialogue_context)": df['ctx_tokens'],
    "当前用户问题 (user_query)"     : df['query_tokens'],
    "黄金答案 (true_a_i)"          : df['answer_tokens'],
    "检索文档总长度"               : df['retrieved_tokens'],
    "历史 + 当前问题"              : df['ctx_tokens'] + df['query_tokens'],
    "完整输入 (ctx + query + docs)": df['ctx_tokens'] + df['query_tokens'] + df['retrieved_tokens'],
}

print("\nToken 长度统计（使用 cl100k_base tokenizer）")
print("═" * 120)
print(f"{'项目':<30} {'平均':>12} {'中位数':>12} {'最大':>14} {'P95':>12} {'总 token 数':>20}")
print("─" * 120)

for name, series in stats.items():
    avg   = series.mean()
    med   = series.median()
    mx    = series.max()
    p95   = series.quantile(0.95)
    total = series.sum()
    
    print(f"{name:<30} "
          f"{avg:12.1f} "
          f"{med:12.0f} "
          f"{mx:14,} "
          f"{p95:12.0f} "
          f"{total:20,}")

print("═" * 120)

# 4. 其他高质量指标
print("\n其他关键指标")
print("-" * 50)
print(f"平均每轮检索文档数       : {df['retrieved_doc_ids'].apply(lambda x: len(eval(x) if isinstance(x,str) else x)).mean():.2f}")
print(f"完整输入 > 8k token 的样本: {(df['ctx_tokens'] + df['query_tokens'] + df['retrieved_tokens'] > 8192).sum():,} 条")

# === 按 dataset 统计 Question_ID 与 query 数量 ===
print("\n数据集分布统计")
print("═" * 80)
print(f"{'Dataset':<20} {'问题数 (Question_ID)':>25} {'查询数 (samples)':>22}")
print("─" * 80)

# 统计每个 dataset 的独特 Question_ID 数和样本数
dist = (
    df.groupby('dataset')
      .agg(
          unique_questions=('Question_ID', 'nunique'),
          total_queries=('query_id', 'nunique')
      )
      .sort_values('total_queries', ascending=False)
)

total_q = dist['unique_questions'].sum()
total_s = dist['total_queries'].sum()

for dataset_name, row in dist.iterrows():
    q_cnt = row['unique_questions']
    s_cnt = row['total_queries']
    print(f"{dataset_name:<20} {q_cnt:>25,} {s_cnt:>22,}")

print("─" * 80)
print(f"{'TOTAL':<20} {total_q:>25,} {total_s:>22,}")
print("═" * 80)

print("\n" + "="*80)
print("                     统计报告完成 — 数据集已准备就绪！")
print("="*80)

多轮数据集加载完成：26,409 条样本

                    MULTI-TURN RAL2M TRAINING SET 完整统计报告
总样本数                  : 26,409
来源数据集数量            : 5

对话轮数分布 (turn_number):
    3 轮对话（含当前轮） → 26,409 条  (100.0%)

加载文档库 D2 用于计算检索文档长度... 成功加载 47,975 条文档

Token 长度统计（使用 cl100k_base tokenizer）
════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
项目                                       平均          中位数             最大          P95            总 token 数
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
历史上下文 (dialogue_context)              192.2          137            938          514            5,075,355
当前用户问题 (user_query)                    17.0           16            101           29              448,920
黄金答案 (true_a_i)                        75.0           44            582          258            1,979,793
检索文档总长度                               884.1          566       

In [2]:
# multiturn_stats_report.py
import pandas as pd
import tiktoken
from collections import Counter
import json

# 加载多轮数据
df = pd.read_json("test_final.jsonl", lines=True)
print(f"多轮数据集加载完成：{len(df):,} 条样本\n")
print("="*80)
print("                    MULTI-TURN RAL2M TESTING SET 完整统计报告")
print("="*80)

# 初始化 tokenizer（GPT系列通用）
enc = tiktoken.get_encoding("cl100k_base")

def count_tokens(text):
    return len(enc.encode(str(text))) if pd.notna(text) else 0

# 1. 基本结构统计
n_samples = len(df)
n_datasets = df['dataset'].nunique()
n_unique_qids = df['Question_ID'].nunique()

print(f"总样本数                  : {n_samples:,}")
print(f"来源数据集数量            : {n_datasets:,}")
print()

# 2. Token 长度统计
df['ctx_tokens'] = df['dialogue_context'].apply(count_tokens)
df['query_tokens'] = df['user_query'].apply(count_tokens)
df['answer_tokens'] = df['true_a_i'].apply(count_tokens)

# retrieved_doc_ids → 实际文档文本（从 D2 加载）
print("加载文档库 D2 用于计算检索文档长度...", end=" ")
d2_docs = {}
with open("D2.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)
        d2_docs[obj["doc_id"]] = obj["text"]
print(f"成功加载 {len(d2_docs):,} 条文档")

def get_retrieved_text(doc_ids):
    if not isinstance(doc_ids, list):
        doc_ids = eval(doc_ids) if isinstance(doc_ids, str) else []
    texts = [d2_docs.get(did, "") for did in doc_ids]
    return " ".join(texts)

df['retrieved_text'] = df['retrieved_doc_ids'].apply(get_retrieved_text)
df['retrieved_tokens'] = df['retrieved_text'].apply(count_tokens)

# 3. 汇总 Token 统计
stats = {
    "历史上下文 (dialogue_context)": df['ctx_tokens'],
    "当前用户问题 (user_query)"     : df['query_tokens'],
    "黄金答案 (true_a_i)"          : df['answer_tokens'],
    "检索文档总长度"               : df['retrieved_tokens'],
    "历史 + 当前问题"              : df['ctx_tokens'] + df['query_tokens'],
    "完整输入 (ctx + query + docs)": df['ctx_tokens'] + df['query_tokens'] + df['retrieved_tokens'],
}

print("\nToken 长度统计（使用 cl100k_base tokenizer）")
print("═" * 120)
print(f"{'项目':<30} {'平均':>12} {'中位数':>12} {'最大':>14} {'P95':>12} {'总 token 数':>20}")
print("─" * 120)

for name, series in stats.items():
    avg   = series.mean()
    med   = series.median()
    mx    = series.max()
    p95   = series.quantile(0.95)
    total = series.sum()
    
    print(f"{name:<30} "
          f"{avg:12.1f} "
          f"{med:12.0f} "
          f"{mx:14,} "
          f"{p95:12.0f} "
          f"{total:20,}")

print("═" * 120)

# 4. 其他高质量指标
print("\n其他关键指标")
print("-" * 50)
print(f"平均每轮检索文档数       : {df['retrieved_doc_ids'].apply(lambda x: len(eval(x) if isinstance(x,str) else x)).mean():.2f}")
print(f"完整输入 > 8k token 的样本: {(df['ctx_tokens'] + df['query_tokens'] + df['retrieved_tokens'] > 8192).sum():,} 条")

# === 按 dataset 统计 Question_ID 与 query 数量 ===
print("\n数据集分布统计")
print("═" * 80)
print(f"{'Dataset':<20} {'问题数 (Question_ID)':>25} {'查询数 (samples)':>22}")
print("─" * 80)

# 统计每个 dataset 的独特 Question_ID 数和样本数
dist = (
    df.groupby('dataset')
      .agg(
          unique_questions=('Question_ID', 'nunique'),
          total_queries=('query_id', 'nunique')
      )
      .sort_values('total_queries', ascending=False)
)

total_q = dist['unique_questions'].sum()
total_s = dist['total_queries'].sum()

for dataset_name, row in dist.iterrows():
    q_cnt = row['unique_questions']
    s_cnt = row['total_queries']
    print(f"{dataset_name:<20} {q_cnt:>25,} {s_cnt:>22,}")

print("─" * 80)
print(f"{'TOTAL':<20} {total_q:>25,} {total_s:>22,}")
print("═" * 80)

print("\n" + "="*80)
print("                     统计报告完成 — 数据集已准备就绪！")
print("="*80)

多轮数据集加载完成：4,842 条样本

                    MULTI-TURN RAL2M TESTING SET 完整统计报告
总样本数                  : 4,842
来源数据集数量            : 5

加载文档库 D2 用于计算检索文档长度... 成功加载 47,975 条文档

Token 长度统计（使用 cl100k_base tokenizer）
════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
项目                                       平均          中位数             最大          P95            总 token 数
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
历史上下文 (dialogue_context)              182.0          139          1,046          483              881,031
当前用户问题 (user_query)                    17.0           16             52           28               82,076
黄金答案 (true_a_i)                        68.2           42            562          216              330,291
检索文档总长度                               756.0          520         24,680         1305            3,660,678
历史 + 当前问题           

In [3]:
# sanity_check_golden_rules.py
import json
from collections import defaultdict

print("=" * 100)
print("GOLDEN RULE SANITY CHECK")
print("1. 每个 Question_ID → 唯一 true_q_i")
print("2. 每个 Question_ID → 正好 3 条记录")
print("3. 每个 Question_ID → 3 条完全不同的 user_query")
print("=" * 100)

for filename in ["train_final_rebuilt.jsonl", "test_final_rebuilt.jsonl"]:
    if not __import__('os').path.exists(filename):
        continue
        
    print(f"\n正在检查 → {filename}")
    print("-" * 80)

    qid_to_true_qi = {}
    qid_to_queries = defaultdict(set)
    qid_to_count = defaultdict(int)
    qid_to_rows = defaultdict(list)

    total_lines = 0

    with open(filename, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            total_lines += 1
            try:
                obj = json.loads(line)
            except:
                print("  JSON 解析失败，跳过一行")
                continue

            qid = obj.get("Question_ID")
            tqi = str(obj.get("true_q_i", "")).strip()
            query = str(obj.get("user_query", "")).strip()

            if not qid:
                print("  发现空 Question_ID，已跳过")
                continue

            qid_to_count[qid] += 1
            qid_to_queries[qid].add(query)
            qid_to_rows[qid].append(obj)

            # 检查 true_q_i 是否一致
            if qid in qid_to_true_qi:
                if qid_to_true_qi[qid] != tqi:
                    print(f"  [ERROR] Question_ID={qid} 有多个 true_q_i！")
            else:
                qid_to_true_qi[qid] = tqi

    total_qids = len(qid_to_count)

    # 统计结果
    rule1_fail = 0  # true_q_i 不唯一
    rule2_fail = 0  # 不是 3 条
    rule3_fail = 0  # query 不唯一

    for qid in qid_to_count:
        count = qid_to_count[qid]
        queries = qid_to_queries[qid]
        tqi = qid_to_true_qi.get(qid, "")

        # Rule 1: 必须有 true_q_i 且唯一（已经在上面检查）
        if not tqi:
            rule1_fail += 1

        # Rule 2: 正好 3 条
        if count != 3:
            rule2_fail += 1

        # Rule 3: 3 条 user_query 必须完全不同
        if len(queries) != 3:
            rule3_fail += 1

    # 输出报告
    print(f"总样本数       : {total_lines:,}")
    print(f"总 Question_ID : {total_qids:,}")

    print(f"\nRule 1: 每个 Question_ID 对应唯一 true_q_i")
    print(f"   → 失败数: {rule1_fail:,} 个")

    print(f"\nRule 2: 每个 Question_ID 正好 3 条记录")
    if rule2_fail == 0:
        print(f"   → 完美！全部都是 3 条")
    else:
        print(f"   → 失败数: {rule2_fail:,} 个")
        from collections import Counter
        cnt = Counter(qid_to_count.values())
        for c in sorted(cnt):
            if c != 3:
                print(f"     → {cnt[c]:,} 个 Question_ID 有 {c} 条记录")

    print(f"\nRule 3: 每个 Question_ID 的 3 条 user_query 完全不同")
    if rule3_fail == 0:
        print(f"   → 完美！所有组内 query 都不同")
    else:
        print(f"   → 失败数: {rule3_fail:,} 个（有重复 query）")

    if rule1_fail + rule2_fail + rule3_fail == 0:
        print("\nALL GOLDEN RULES PASSED! 数据完美，可直接训练！")
    else:
        print(f"\n存在 {rule1_fail + rule2_fail + rule3_fail} 个问题，请修复")

print("\n" + "="*100)
print("检查完成！")

GOLDEN RULE SANITY CHECK
1. 每个 Question_ID → 唯一 true_q_i
2. 每个 Question_ID → 正好 3 条记录
3. 每个 Question_ID → 3 条完全不同的 user_query

检查完成！


In [4]:
# ultimate_sanity_check.py
# 2025 终极数据体检脚本 —— 任何问题都逃不过

import json
import pandas as pd
from collections import defaultdict, Counter
import re

print("=" * 110)
print("ULTIMATE SANITY CHECK 2025 — 全面体检，零容忍")
print("=" * 110)

# 1. 加载 D1（黄金问题库）
print("正在加载 D1.jsonl（黄金问题库）...")
try:
    d1 = pd.read_json("D1.jsonl", lines=True)
    d1_true_qi_set = set(d1['Q'].astype(str).str.strip())
    print(f"D1.jsonl 加载成功，共 {len(d1_true_qi_set):,} 个唯一 true_q_i")
except Exception as e:
    print(f"D1.jsonl 加载失败: {e}")
    d1_true_qi_set = set()

# 2. 检查所有数据文件
files = ["train_final.jsonl", "test_final.jsonl"]
files = [f for f in files if __import__("os").path.exists(f)]

for filename in files:
    print(f"\n{'='*45} 检查 {filename} {'='*45}")

    # 统计容器
    qid_stats = defaultdict(lambda: {"count": 0, "queries": set(), "true_qi": None})
    all_query_ids = set()
    all_user_queries = Counter()           # 全局 user_query 去重计数
    query_to_qids = defaultdict(set)       # 同一个 query 出现在哪些 Question_ID
    total_lines = 0
    missing_fields = 0
    true_qi_not_in_d1 = 0
    dialogue_format_errors = 0
    total_with_context = 0

    with open(filename, "r", encoding="utf-8") as f:
        for line_num, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            total_lines += 1
            try:
                obj = json.loads(line)
            except:
                print(f"  [ERROR] 第 {line_num} 行 JSON 解析失败")
                continue

            # 必填字段检查
            qid = str(obj.get("Question_ID", "")).strip()
            query_id = obj.get("query_id")
            true_qi = str(obj.get("true_q_i", "")).strip()
            user_query = str(obj.get("user_query", "")).strip()
            dialogue_context = str(obj.get("dialogue_context", "")).strip()

            if not all([qid, query_id, true_qi, user_query]):
                missing_fields += 1
                continue

            # 全局统计
            all_user_queries[user_query] += 1
            query_to_qids[user_query].add(qid)

            # query_id 唯一性
            if query_id in all_query_ids:
                print(f"  [ERROR] 重复 query_id: {query_id} (第 {line_num} 行)")
            all_query_ids.add(query_id)

            # Question_ID 统计
            qid_stats[qid]["count"] += 1
            qid_stats[qid]["queries"].add(user_query)
            if qid_stats[qid]["true_qi"] is None:
                qid_stats[qid]["true_qi"] = true_qi
            elif qid_stats[qid]["true_qi"] != true_qi:
                print(f"  [ERROR] Question_ID={qid} 有多个 true_q_i！")

            # true_q_i 是否在 D1
            if true_qi not in d1_true_qi_set:
                true_qi_not_in_d1 += 1

            # dialogue_context 格式检查（必须是 2 User + 2 Assistant）
            if dialogue_context:
                total_with_context += 1
                user_cnt = len(re.findall(r'\bUser\s*:', dialogue_context, flags=re.I))
                assistant_cnt = len(re.findall(r'\bAssistant\s*:', dialogue_context, flags=re.I))
                if user_cnt != 2 or assistant_cnt != 2:
                    dialogue_format_errors += 1
                    if dialogue_format_errors <= 5:
                        print(f"  [ERROR] dialogue_context 格式错误 (U={user_cnt}, A={assistant_cnt}):")
                        print(f"     → {dialogue_context[:150]}{'...' if len(dialogue_context)>150 else ''}")

    total_qids = len(qid_stats)

    # 汇总问题
    not_3_entries = sum(1 for s in qid_stats.values() if s["count"] != 3)
    duplicate_query_in_qid = sum(1 for s in qid_stats.values() if len(s["queries"]) < s["count"])
    global_duplicate_queries = sum(1 for cnt in all_user_queries.values() if cnt > 1)

    print(f"\n数据概览：")
    print(f"   总样本数             : {total_lines:,}")
    print(f"   总 Question_ID        : {total_qids:,}")
    print(f"   总 query_id          : {len(all_query_ids):,}")

    print(f"\n核心检查结果：")
    print(f" 1. 每个 Question_ID 有唯一 true_q_i        : ", "PASSED" if not any(
        s["true_qi"] is None for s in qid_stats.values()) else "FAILED")
    print(f" 2. 每个 Question_ID 正好 3 条记录          : ", "PASSED" if not_3_entries == 0 else f"FAILED ({not_3_entries} 个不等于3)")
    print(f" 3. 每个 Question_ID 内 3 条 user_query 不同 : ", "PASSED" if duplicate_query_in_qid == 0 else f"FAILED ({duplicate_query_in_qid} 个有重复)")
    print(f" 4. 所有 query_id 全局唯一                 : ", "PASSED" if len(all_query_ids) == total_lines else "FAILED")
    print(f" 5. 所有 true_q_i 都在 D1.jsonl 中           : ", "PASSED" if true_qi_not_in_d1 == 0 else f"FAILED ({true_qi_not_in_d1} 个缺失)")
    print(f" 6. 多轮样本 dialogue_context 格式正确       : ", "PASSED" if dialogue_format_errors == 0 else f"FAILED ({dialogue_format_errors} 条错误)")

    print(f"\n新增强力检查：")
    print(f" 7. 全局 user_query 重复情况               : ", end="")
    if global_duplicate_queries == 0:
        print("PASSED（完全无重复）")
    else:
        print(f"FAILED — 发现 {global_duplicate_queries:,} 个重复 user_query！")
        print("   重复最严重的几个例子：")
        for q, cnt in all_user_queries.most_common(10):
            if cnt > 1:
                qids = query_to_qids[q]
                print(f"     → \"{q[:80]}{'...' if len(q)>80 else ''}\" 出现了 {cnt} 次，涉及 Question_ID: {list(qids)[:5]}")

    # 最终判决
    all_pass = (
        not_3_entries == 0 and
        duplicate_query_in_qid == 0 and
        len(all_query_ids) == total_lines and
        true_qi_not_in_d1 == 0 and
        dialogue_format_errors == 0 and
        global_duplicate_queries == 0
    )

    print(f"\n{'='*30} 最终结论 {'='*30}")
    if all_pass:
        print("ALL CHECKS PASSED! 数据完美无瑕，可以直接训练！")
    else:
        print("存在严重问题，必须修复后再使用！")

print("\n" + "="*110)
print("终极检查完成！")

ULTIMATE SANITY CHECK 2025 — 全面体检，零容忍
正在加载 D1.jsonl（黄金问题库）...
D1.jsonl 加载成功，共 10,411 个唯一 true_q_i

============================================= 检查 train_final.jsonl =============================================

数据概览：
   总样本数             : 26,409
   总 Question_ID        : 8,803
   总 query_id          : 26,409

核心检查结果：
 1. 每个 Question_ID 有唯一 true_q_i        :  PASSED
 2. 每个 Question_ID 正好 3 条记录          :  PASSED
 3. 每个 Question_ID 内 3 条 user_query 不同 :  PASSED
 4. 所有 query_id 全局唯一                 :  PASSED
 5. 所有 true_q_i 都在 D1.jsonl 中           :  PASSED
 6. 多轮样本 dialogue_context 格式正确       :  PASSED

新增强力检查：
 7. 全局 user_query 重复情况               : PASSED（完全无重复）

============================== 最终结论 ==============================
ALL CHECKS PASSED! 数据完美无瑕，可以直接训练！

============================================= 检查 test_final.jsonl =============================================

数据概览：
   总样本数             : 4,842
   总 Question_ID        : 1,614
   总 query_id          : 4,842

核心检查结果：
 1. 每个

In [5]:
# # FINAL_CORRECT_PIPELINE.py
# # 顺序：先替换成真实文本 → 再搬数据 → 再统一建 D2 → 再转回 ID
# import pandas as pd
# import numpy as np
# import json
# import os

# np.random.seed(42)

# # ====================== 1 先加载原始数据 ======================
# print("1. 加载原始数据...")
# train_raw = pd.read_json("train_final.jsonl", lines=True)
# test_raw  = pd.read_json("test_final_2.jsonl", lines=True)

# # ======================2 加载各自文档库（完全隔离）======================
# def load_docs(path):
#     docs = {}
#     with open(path, "r", encoding="utf-8") as f:
#         for line in f:
#             if line.strip():
#                 obj = json.loads(line.strip())
#                 docs[obj["doc_id"]] = obj["text"].strip()
#     return docs

# print("2. 加载各自文档库...")
# d2_train = load_docs("D2.jsonl")        # train 专用
# d2_test  = load_docs("D2_test.jsonl")   # test 专用
# print(f"   D2.jsonl      : {len(d2_train):,} docs")
# print(f"   D2_test.jsonl : {len(d2_test):,}  docs")

# # ======================3 先把 doc_id 换成真实文本（各自用各自的 D2）======================
# def replace_with_real_text(df, doc_dict, name):
#     print(f"3. {name}：retrieved_doc_ids → 真实文档文本...")
#     def parse_ids(x):
#         if pd.isna(x): return []
#         if isinstance(x, list): return x
#         try: return json.loads(str(x).replace("'", '"'))
#         except: return eval(str(x)) if str(x).startswith('[') else []

#     def get_text(ids):
#         parts = []
#         for did in parse_ids(ids):
#             text = doc_dict.get(did, "[MISSING]")
#             parts.append(f"[DOC:{did}]\n{text}")
#         return "\n\n".join(parts) if parts else ""

#     df = df.copy()
#     df['retrieved_documents'] = df['retrieved_doc_ids'].apply(get_text)
#     return df

# train_with_text = replace_with_real_text(train_raw, d2_train, "train")
# test_with_text  = replace_with_real_text(test_raw,  d2_test,  "test")

# # ======================4 现在才搬数据！带着真实文本一起搬！======================
# print("\n4. 搬一半 hagrid 到 train（带着真实文本一起搬）...")
# hagrid_qids = test_with_text[test_with_text['dataset'] == 'hagrid']['Question_ID'].unique()
# move_qids   = np.random.choice(hagrid_qids, size=len(hagrid_qids)//2, replace=False)

# move_df   = test_with_text[test_with_text['Question_ID'].isin(move_qids)].copy()
# remain_df = test_with_text[~test_with_text['Question_ID'].isin(move_qids)].copy()

# # 给搬过去的分配全新连续 Question_ID
# max_id = pd.to_numeric(train_with_text['Question_ID'], errors='coerce').max()
# next_id = int(max_id) + 1 if pd.notna(max_id) else 1
# new_ids = [str(next_id + i) for i in range(len(move_qids))]
# id_map = dict(zip(move_qids, new_ids))

# move_df['Question_ID'] = move_df['Question_ID'].map(id_map)
# if 'query_id' in move_df.columns:
#     move_df['query_id'] = move_df['query_id'].str.replace(
#         r'^[^_]+', lambda m: id_map.get(m.group(0).split('_p')[0], m.group(0)), regex=True
#     )

# # 合并成最终 train
# final_train = pd.concat([train_with_text, move_df], ignore_index=True)
# final_test  = remain_df

# print(f"   成功搬迁 {len(move_qids):,} 个 hagrid 问题（{len(move_df):,} 条样本）")
# print(f"   新 train 大小 : {len(final_train):,}")
# print(f"   新 test  大小 : {len(final_test):,}")

# # ======================5 统一去重所有真实文档文本 → 建共享 D2.jsonl ======================
# print("\n5. 统一去重所有真实文档内容 → 生成共享 D2.jsonl...")
# all_texts = pd.concat([
#     final_train['retrieved_documents'],
#     final_test['retrieved_documents']
# ], ignore_index=True)

# unique_texts = {}
# doc_list = []
# doc_id = 0
# text_to_id = {}

# for block in all_texts:
#     if not block or pd.isna(block): continue
#     for part in block.split("[DOC:")[1:]:
#         if "]\n" not in part: continue
#         doc_text = part.split("]\n", 1)[1].strip()
#         if doc_text not in text_to_id:
#             new_id = f"D{doc_id}"
#             text_to_id[doc_text] = new_id
#             doc_list.append({"doc_id": new_id, "text": doc_text})
#             doc_id += 1

# with open("D2.jsonl", "w", encoding="utf-8") as f:
#     for d in doc_list:
#         json.dump(d, f, ensure_ascii=False)
#         f.write("\n")
# print(f"   共享文档库建成：{len(doc_list):,} 条唯一文档 → D2.jsonl")

# # ======================6 把真实文本重新映射回新 doc_id ======================
# def text_to_new_ids(block):
#     if not block or pd.isna(block): return []
#     ids = []
#     for part in block.split("[DOC:")[1:]:
#         if "]\n" not in part: continue
#         old_id = part.split("]", 1)[0]
#         doc_text = part.split("]\n", 1)[1].strip()
#         ids.append(text_to_id[doc_text])
#     ids = list(dict.fromkeys(ids))  # 保持顺序去重
#     return ids

# print("\n6. 映射回新 doc_id...")
# final_train['retrieved_doc_ids'] = final_train['retrieved_documents'].apply(text_to_new_ids)
# final_test['retrieved_doc_ids']  = final_test['retrieved_documents'].apply(text_to_new_ids)

# # 删除中间列
# final_train = final_train.drop(columns=['retrieved_documents'])
# final_test  = final_test.drop(columns=['retrieved_documents'])

# # ======================7 保存最终文件 ======================
# final_train.to_json("train_final_CLEAN.jsonl", orient="records", lines=True, force_ascii=False)
# final_test.to_json("test_final_CLEAN.jsonl",  orient="records", lines=True, force_ascii=False)

# print("\n" + "="*80)
# print("MISSION ACCOMPLISHED! 全部正确完成！")
# print("="*80)
# print("   D2.jsonl                  → 共享唯一文档库（去重后）")
# print("   train_final_CLEAN.jsonl   → 最终训练集（已搬 hagrid + 新 doc_id）")
# print("   test_final_CLEAN.jsonl    → 最终测试集（干净）")
# print(f"   总唯一文档 : {len(doc_list):,} 条")
# print(f"   train 样本 : {len(final_train):,}")
# print(f"   test  样本 : {len(final_test):,}")
# print("="*80)
# print("现在可以直接开训了！文档不冲突、ID不冲突、搬数据顺序错误的问题全部解决！")

In [6]:
# final_train.to_json("train_final_updated_cleaned.jsonl", orient="records", lines=True, force_ascii=False)

# final_test.to_json("test_final_updated_cleaned.jsonl", orient="records", lines=True, force_ascii=False)
